<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_pinn_particle_diffusion_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.1 — A Particle in a Battery Cell

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L10.1 · Ionic Diffusion and Charge Conservation**

A lithium-ion cell is discharged at 5 A for an hour. In its negative electrode
the lithium sits inside particles of graphite, a few micrometres across, and
has to diffuse to the surface of a particle before it can leave. How far does
the surface run ahead of the centre, and is there any lithium left at the
surface at the end? It is diffusion in a sphere - the single particle model
of L10.1. You solve it three ways - an exact series, finite volumes, and a
physics-informed network - and compare them on accuracy and on time. Then the
same network finds the diffusivity from the surface concentration, as a
battery management system would have to.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the exact solution, and finite volumes for an agreed 30 mol/m³ | — |
| **5** | a PINN with the start built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: the diffusivity from the surface concentration | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About two minutes on a CPU. Concentrations are printed in mol/m³ without
decimals; other numbers with at most two.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex10.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

An LG M50 cell, a 21700 of 5 Ah, is discharged at 1C: 5 A for one hour. In the
single particle model its negative electrode is **one representative particle
of graphite**, a sphere of radius 5.86 µm. Lithium leaves through the surface
at a constant rate, and the lithium inside has to diffuse outwards to replace
it. What is the concentration at every radius over the hour - and how far
does the surface, which is what the reaction sees, run ahead of the centre?

## 2 · The data

| | |
|---|---|
| particle | graphite, radius $R$ = 5.86 µm, diffusivity $D_s$ = 3.3e-14 m²/s |
| start | $c_0$ = 29 866 mol/m³ everywhere (0.90 of the maximum, 33 133 mol/m³) |
| discharge | $I$ = 5 A for $t_{\mathrm{end}}$ = 3600 s |
| electrode | 85.2 µm thick, 0.1027 m² of sheet, 75 % graphite by volume |
| surface flux | $j = I / (F\,a_s\,A\,L)$ = 1.54e-05 mol/(m² s), leaving |

The values are those of Chen et al. (2020) for this cell, as distributed with
PyBaMM.

## 3 · The physics

Inside the particle the lithium diffuses (L10.1's *Diffusion Inside a
Particle*); it is the heat equation of L8.2 in a sphere:

$$c_t = D_s\Big(c_{rr} + \frac{2}{r}\,c_r\Big), \qquad -D_s\,c_r = j \ \text{at}\ r = R, \qquad c_r = 0 \ \text{at}\ r = 0$$

Two things are new. The term $2/r$ comes from the sphere, and it is singular
at the centre. And the surface condition is on the **flux**, not on the
value: the current fixes how fast lithium leaves, and nothing fixes the
concentration anywhere.

The time for lithium to cross the particle is $R^2/D_s$ = 1041 s, a third of
the discharge, so the particle is neither uniform nor frozen.

**Scaled units.** Radius over $R$, time over the hour, and the concentration
lost over $c_{\mathrm{ref}} = j\,t_{\mathrm{end}}/R$ = 9476 mol/m³:

$$u = \frac{c_0 - c}{c_{\mathrm{ref}}}, \qquad u_\tau = C\Big(u_{rr} + \frac{2}{r}\,u_r\Big), \qquad C\,u_r = 1 \ \text{at}\ r = 1, \qquad C = \frac{D_s\,t_{\mathrm{end}}}{R^2} = 3.46$$

with $u = 0$ at $\tau = 0$. **The mean of $u$ over the sphere is exactly
$3\tau$**, whatever the diffusivity: what has left the particle is the current
times the time. Everything below is solved in these units and printed in
mol/m³.

**Why only the concentration is solved.** The single particle model is
usually written as a set of equations, but only the diffusion above is a
differential equation. The rest is algebra, applied once it is solved.
Charge conservation is trivial here - identical particles share the current
evenly, so $j$ is known from $I$ before anything is computed - and
Butler-Volmer (L10.1) turns $j$ into an overpotential $\eta$ at each
surface. The terminal voltage then reads the open-circuit curves $U$ at the
two surface concentrations,

$$V(t) = U_p\big(c_p^{\mathrm{surf}}\big) - U_n\big(c_n^{\mathrm{surf}}\big) + \eta_p - \eta_n,$$

one line of postprocessing. Section 6 walks the same chain backwards, from
the measured voltage to the surface concentration.

**The electrode, node by node.** The real electrode is many particles, and
the full Doyle-Fuller-Newman model (section 9's first mini project) meshes
the thickness too: one representative particle at every node. Lithium never
diffuses from particle to particle. Each particle trades lithium with the
electrolyte at its own surface; between particles the lithium travels as
ions in the electrolyte, and the electrons through the solid. The second
drawing below shows that mesh on a harder cell, the A123 LFP of 2.3 Ah
(PyBaMM's Prada2013 parameters): particles of 50 nm instead of 5.86 µm, yet
almost as slow to cross ($R^2/D_s$ = 424 s at $D_s$ = 5.9e-18 m²/s), and an
open-circuit curve that is a plateau - about 3.40 V, moving some 10 mV
between 10 and 70 % of charge - so there the voltage says almost nothing
about the surface concentration, and section 6's chain backwards goes
nearly blind.

The cell below prints the data, draws the particle and the electrode it
stands for, draws the whole cell with the full model's mesh, and draws the
exact solution.

In [ ]:
pb.describe_problem()
pb.draw_particle(); plt.show()                               # the electrode, and the one particle that stands for it
pb.draw_sandwich(); plt.show()                               # the whole cell, and the full model's mesh (an LFP cell)
r = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for t in (300, 900, 1800, 3600):
    ax.plot(r * pb.R_PARTICLE * 1e6, pb.concentration(pb.exact(r, t / pb.T_END)), label=f"{t} s")
ax.set_xlabel("radius  [µm]"); ax.set_ylabel("lithium concentration  [mol/m³]")
ax.legend(frameon=False, title="exact solution at"); plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, a surface flux of 1.54e-05
mol/(m² s), a diffusion time of 1041 s and a scaled diffusivity of 3.46; the
two drawings; and four profiles, each lower at the surface than at the
centre, falling as the hour goes on.

---

## 4 · The reference, and finite volumes

**The reference** is exact: the classical series solution for a sphere with a
constant flux through its surface, `pb.exact(r, tau)`.

**Finite volumes.** The sphere is cut into shells of equal thickness. Each
shell keeps count of its lithium: what crosses its inner face comes in, what
crosses its outer face goes out, and the last shell loses $j$ through the
surface. The faces are weighted by $r^2$, so the $2/r$ term never appears and
the centre needs no special care. In time it is Crank-Nicolson, one small
tridiagonal solve per step.

**The agreed accuracy is 30 mol/m³**, at every radius and at every one of the
36 instants a battery management system would log, 100 s apart: a thousandth
of what the graphite can hold. The cell below finds the coarsest grid that
meets it.

In [ ]:
INSTANTS = np.linspace(0, 1, 37)[1:]                          # the 36 logged instants, 100 s apart

def fvm_particle(n, steps, c=pb.C_SCALED):
    # n shells, `steps` time steps; returns the shell centres and u at the 36 instants (shells, then the surface)
    h = 1.0 / n
    edge = np.arange(n + 1) * h; centre = (edge[:-1] + edge[1:]) / 2
    vol = (edge[1:] ** 3 - edge[:-1] ** 3) / 3                 # shell volume over 4 pi
    k = edge[1:-1] ** 2 / h                                    # face area over the distance between centres
    diag = np.zeros(n); diag[:-1] -= k; diag[1:] -= k
    A = sp.diags(1 / vol) @ sp.diags([k, diag, k], [-1, 0, 1]) * c   # the diffusion between neighbouring shells
    b = np.zeros(n); b[-1] = 1.0 / vol[-1]                     # the surface flux, C u_r = 1, into the last shell
    dt = 1.0 / steps; I = sp.identity(n)
    lu = spla.splu((I - 0.5 * dt * A).tocsc()); B = (I + 0.5 * dt * A).tocsr()   # Crank-Nicolson
    u = np.zeros(n); out = []
    for step in range(1, steps + 1):
        u = lu.solve(B @ u + dt * b)
        if step % (steps // 36) == 0:
            out.append(np.r_[u, u[-1] + 0.5 * h / c])          # the surface: half a shell further, at the known gradient
    return np.r_[centre, 1.0], np.array(out)

def worst_error(r, U):
    # the worst error against the exact solution over the radii and the 36 instants, in mol/m3
    return pb.C_REF * np.abs(U - pb.exact(r[None, :], INSTANTS[:, None])).max()

TARGET = 30.0                                                 # mol/m3: the agreed accuracy
for n, steps in ((10, 36), (20, 72), (40, 144), (80, 288)):
    t0 = time.perf_counter(); r_f, U = fvm_particle(n, steps); sec = time.perf_counter() - t0
    err = worst_error(r_f, U)
    print(f"  {n:3d} shells, {steps:3d} steps: surface at the end {pb.concentration(U[-1, -1]):.0f} mol/m3, "
          f"worst error {err:.0f} mol/m3, {1000 * sec:.0f} ms" + ("   <- meets 30 mol/m3" if err <= TARGET else ""))
N_FVM, STEPS = 40, 144                                        # the coarsest grid that meets it

**What you should see.** The error falling with every halving of the shells
and the step - 136, 52, 29 and 15 mol/m³ - and **40 shells with 144
steps the first under 30 mol/m³**, in a few hundredths of a second. The
surface ends the hour at about 890 mol/m³: nearly empty, while the centre
still holds 2260.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(r, \tau)$ and returns the scaled loss $u$. It is four
layers of 32 tanh neurons, trained on 2000 collocation points, more of them
early in the hour.

**The start is built in.** The trial function

$$\hat u(r, \tau) = S\;\tau\;\mathcal{N}(r, \tau)$$

is exactly zero at $\tau = 0$ whatever the network does. $S$ = 3 is the size
of the answer at the end of the hour.

**The residual is multiplied by $r$.** Written as
$r\,\hat u_\tau - C\,(r\,\hat u_{rr} + 2\,\hat u_r)$ it has no $1/r$ in it, and
at the centre it reduces to $\hat u_r = 0$: the symmetry condition, for free.

**Two more loss terms.** The surface flux, $C\,\hat u_r = 1$ at $r = 1$, with
the weight $w_S$ = 10. And the **mass balance**: the mean of $\hat u$ over the
sphere must be $3\tau$, with the weight $w_M$ = 1000. L8.1 used a balance like
that as a check after training; here it is in the loss, because a network
that is a little late with the surface flux at the very start loses a little
lithium and never gets it back.

**TODO 1** is the trial function; **TODO 2** the residual.

In [ ]:
SCALE = 3.0                                                   # the size of the answer at the end of the hour
class Particle(torch.nn.Module):                              # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: r, tau in
    def forward(self, p):
        tau = p[:, 1:2]
        return SCALE * tau * self.net(p)                      # the trial function: zero at the start

In [ ]:
def residual(model, p, c):
    u = model(p)                                              # the network's scaled loss of lithium
    g = grad(u, p)                                            # first derivatives by autograd; columns r, tau
    r = p[:, 0:1]
    return r * g[:, 1:2] - c * (r * d2(u, p, 0) + 2 * g[:, 0:1])   # the diffusion equation, multiplied by r

In [ ]:
W_S, W_M = 10.0, 1000.0                                       # the weights of the surface flux and of the mass balance
pts = to_tensor(pb.sample_particle(2000), requires_grad=True) # collocation points (r, tau), differentiable
surf = to_tensor(pb.surface_points(200), requires_grad=True)  # 200 instants on the surface, differentiable
q_pts, q_w, q_target = (to_tensor(a) for a in pb.mass_quadrature())   # the mass balance: points, weights, 3 tau

def flux(model, c):
    return c * grad(model(surf), surf)[:, 0:1] - 1.0          # the surface condition C u_r = 1, by autograd

def mass(model):
    mean = (model(q_pts) * q_w).reshape(len(q_target), -1).sum(1, keepdim=True)   # the mean of u over the sphere
    return mean - q_target                                    # it must be 3 tau at every instant

def loss_of(model, c):
    # the composite loss: the equation, the surface flux and the mass balance
    return residual(model, pts, c).pow(2).mean() + W_S * flux(model, c).pow(2).mean() + W_M * mass(model).pow(2).mean()

model = Particle().to(DEVICE)                                 # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, lambda: loss_of(model, pb.C_SCALED), adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
t0 = time.perf_counter(); r_f, U = fvm_particle(N_FVM, STEPS); t_fvm = time.perf_counter() - t0   # finite volumes
grid = to_tensor(np.array([[r, t] for t in INSTANTS for r in r_f]))   # the same radii and instants for the network
def pinn_on_grid(net=None):
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy((net or model)(grid)).reshape(len(INSTANTS), -1)   # u at the 36 instants
P = pinn_on_grid()
t_pin = middle(pinn_on_grid, reps=3)
E = pb.exact(r_f[None, :], INSTANTS[:, None])                 # the exact solution on the same radii and instants
e_fvm, e_pin = pb.C_REF * np.abs(U - E).max(), pb.C_REF * np.abs(P - E).max()

print(f"{'':28s}{'surface at the end':>20s}{'centre':>12s}{'worst error':>14s}{'the hour':>11s}{'training':>10s}")
print(f"  {'exact series':26s}{pb.concentration(E[-1, -1]):11.0f} mol/m3{pb.concentration(pb.exact(0.0, 1.0)):12.0f}{'-':>14s}{'-':>11s}{'-':>10s}")
print(f"  {'finite volumes, 40 shells':26s}{pb.concentration(U[-1, -1]):11.0f} mol/m3{pb.concentration(U[-1, 0]):12.0f}{e_fvm:7.0f} mol/m3{t_fvm:9.3f} s{'-':>10s}")
print(f"  {'PINN, 4 x 32':26s}{pb.concentration(P[-1, -1]):11.0f} mol/m3{pb.concentration(P[-1, 0]):12.0f}{e_pin:7.0f} mol/m3{t_pin:9.3f} s{train_time:8.0f} s")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for k, col in ((2, "tab:blue"), (17, "tab:green"), (35, "tab:red")):
    axes[0].plot(r_f * pb.R_PARTICLE * 1e6, pb.concentration(E[k]), color=col, lw=2.5, label=f"exact, {100 * (k + 1)} s")
    axes[0].plot(r_f * pb.R_PARTICLE * 1e6, pb.concentration(P[k]), "k:", lw=2)
axes[0].plot([], [], "k:", lw=2, label="PINN")
axes[0].set_xlabel("radius  [µm]"); axes[0].set_ylabel("concentration  [mol/m³]"); axes[0].legend(frameon=False, fontsize=8)
t_s = INSTANTS * pb.T_END
axes[1].semilogy(t_s, pb.C_REF * np.abs(U - E).max(axis=1), label="finite volumes, 40 shells")
axes[1].semilogy(t_s, pb.C_REF * np.abs(P - E).max(axis=1), label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 30 mol/m³")
axes[1].set_xlabel("t  [s]"); axes[1].set_ylabel("worst error over the particle  [mol/m³]"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex10.1_report.png"), dpi=150); plt.show()

**What you should see.** All three put the surface at about 890 mol/m³ at the
end of the hour and the centre at about 2260: the surface is nearly empty
while the centre still holds more than twice as much. Finite volumes are
within 29 mol/m³ of the exact solution and the network within about 21
mol/m³, both inside the agreed 30. Finite volumes take a few hundredths of a
second and no training; the network needs about 45 s of training on a CPU.

---

## 6 · The inverse problem: the diffusivity from the surface concentration

A cell ages, and its diffusivity falls. It cannot be measured inside a
working cell - but a battery management system sees the voltage, and through
the open-circuit curve the voltage gives the concentration at the particle's
**surface**. Suppose it logged that every 100 s over the discharge, with a
noise of 50 mol/m³. Can the same network find $D_s$?

It is L10.1's *What You Can Recover From a Voltage Curve*. One number is
trained beside the weights: $C = e^{\beta}$, an exponential so that it cannot
go negative. The loss gains one term, the misfit to the readings, with the
weight $w_D$ = 1000; the residual and the surface flux are the ones above, now
with the trainable $C$. $\beta$ starts at $C$ = 1.73, half the true value. The
readings are **synthetic**: the exact solution, sampled, with noise added.

**What makes it identifiable.** The mass balance does not contain $D_s$: it
fixes the mean. The readings are of the surface. The diffusivity is what
relates the two - a slow particle has a surface far below its mean.

In [ ]:
tau_s, c_s = pb.surface_readings()                            # 37 instants, mol/m3, with noise
S_pts = to_tensor(np.c_[np.ones(len(tau_s)), tau_s])          # where and when they were read: the surface
S_val = to_tensor(((pb.C_START - c_s) / pb.C_REF).reshape(-1, 1))   # the readings, scaled

class Inverse(torch.nn.Module):                               # the particle's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.particle = Particle()                            # a fresh network, the same trial function
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(pb.C_SCALED / 2))))   # C = exp(beta), starts at half
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 1000.0                                                  # the weight of the readings
def inverse_loss():
    c = torch.exp(inv.beta)                                   # the diffusivity being identified, always positive
    data = (inv.particle(S_pts) - S_val).pow(2).mean()        # the misfit at the surface
    return loss_of(inv.particle, c) + W_D * data              # the same loss, and the readings
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
C_found = float(torch.exp(inv.beta))
D_found = C_found * pb.R_PARTICLE ** 2 / pb.T_END             # back to m^2/s
print(f"diffusivity: found {D_found:.2e} m^2/s, true {pb.D_SOLID:.2e} m^2/s, "
      f"off by {100 * (D_found / pb.D_SOLID - 1):+.2f} %   ({inv_time:.0f} s)")

**What you should see.** The diffusivity found within about 0.4 % of the
true 3.3e-14 m²/s, in under a minute on a CPU - from 37 noisy readings of the
surface and a start that was half the true value.

---

## 7 · What the notebook says

* **The surface runs ahead of the centre.** After an hour at 1C the surface of
  the particle is nearly empty, at about 890 mol/m³, while the centre still
  holds 2260. The cell's voltage follows the surface, so the cell looks empty
  before all its lithium has arrived.
* **All three methods meet the agreed 30 mol/m³.** The series is exact,
  finite volumes need 40 shells, and the network is within about 21.
* **For the forward problem, finite volumes win on time.** They give the hour
  in hundredths of a second with no training. A forward simulation of a cell
  is a job for a solver; PyBaMM does it for the whole cell.
* **The mass balance belongs in the loss.** The mean concentration is known
  from the current alone. Without that term the network loses a little
  lithium in the first instants and carries the error to the end.
* **The inverse problem is where the network earns its training.** The same
  code, with one more trainable number and one more loss term, returned the
  diffusivity from readings of the surface alone. That is what a battery
  management system needs and cannot measure.
* **The readings must show the surface lagging.** At a very low current the
  particle stays uniform, the surface equals the mean, and the readings say
  nothing about the diffusivity.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex10.1_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L10.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which of the five fields of a cell model did this notebook solve for, on which length scale, and how far was the surface below the centre at the end of the discharge? (-> L10.1 Q1)": """
""",
    "The notebook's particle is the single particle model. What does that model assume about the electrode, and at which currents would you stop trusting it? (-> L10.1 Q2)": """
""",
    "The diffusivity rises with temperature. Using the time R²/D_s = 1041 s, say what a warmer cell does to the gap between surface and centre, and why the thermal and the electrochemical model are solved together. (-> L10.1 Q3)": """
""",
    "Which scales made every number in the notebook of order one, and what did the network do in section 6 that the finite-volume solver could not do directly? (-> L10.1 Q4)": """
""",
    "To conclude: finite volumes gave the hour in hundredths of a second. For which task would you use the network, and what would you tell a battery engineer about how far to trust the diffusivity it found? (-> L10.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | surface at the end | centre at the end | worst error | the hour | training |", "|---|---|---|---|---|---|",
            f"| exact series | {pb.concentration(E[-1, -1]):.0f} mol/m3 | {pb.concentration(pb.exact(0.0, 1.0)):.0f} mol/m3 | - | - | - |",
            f"| finite volumes, 40 shells, 144 steps | {pb.concentration(U[-1, -1]):.0f} mol/m3 | {pb.concentration(U[-1, 0]):.0f} mol/m3 | {e_fvm:.0f} mol/m3 | {t_fvm:.3f} s | - |",
            f"| PINN, 4 x 32 | {pb.concentration(P[-1, -1]):.0f} mol/m3 | {pb.concentration(P[-1, 0]):.0f} mol/m3 | {e_pin:.0f} mol/m3 | {t_pin:.3f} s | {train_time:.0f} s |",
            "", f"Inverse problem: diffusivity found {D_found:.2e} m^2/s, true {pb.D_SOLID:.2e} m^2/s, "
                f"off by {100 * (D_found / pb.D_SOLID - 1):+.2f} %, in {inv_time:.0f} s"]
    out = ["# Ex_10.1 — A Particle in a Battery Cell", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex10.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex10.1_report.md into Ex10.1_report.pdf, with any figure
# saved as Ex10.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex10.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex10.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex10.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex10.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP10.1A · The full model, where the SPMe stops

**The problem.** The set solved one particle - the single particle model, which holds to about
1C. Beyond that the cell needs the Doyle-Fuller-Newman model: lithium diffusing in
the electrolyte across the whole cell as well as in the particles, a particle
at every point of each electrode, the two coupled by the reaction.

| | |
|---|---|
| cell | the LG M50 21700 of this set, PyBaMM's Chen2020 parameters, 5 Ah, 25 °C |
| test | constant-current discharge from full to the 2.5 V cut-off |
| rates | 1C, 2C and 3C |

**Why a plain PINN is not enough.** It is several coupled PDEs on domains of different dimension - the cell's
thickness and each particle's radius - with fluxes, not values, at most
boundaries, and at 3C the electrolyte runs out near the separator. One network
over all of it is the wrong shape.

**What you build.** Coupled PINNs for the Doyle-Fuller-Newman model: networks for the
electrolyte concentration and potential across the cell and for the particle
concentration, joined by the reaction term, and the terminal voltage from
them.

**The ground truth you get.** `MP10.1A_truth_1C.npz`, `_2C.npz` and `_3C.npz`, from
`tools/miniprojects/ex101_truth.py`: PyBaMM's DFN on a mesh four times finer
than its default, with the voltage, the capacity and the electrolyte
concentration over the discharge.

**Worked example.** 

| rate | DFN capacity, meshes ×1 / ×2 / ×4 | voltage, ×2 against ×4 | SPMe capacity | SPMe voltage off by |
|---|---|---|---|---|
| 1C | 4.938 / 4.938 / 4.938 Ah | 1.3 mV | 4.939 Ah | 11 mV |
| 2C | 4.731 / 4.731 / 4.731 Ah | 2.8 mV | 4.757 Ah | 62 mV |
| 3C | 2.303 / 2.328 / 2.335 Ah | 10.2 mV | 0.247 Ah | 1128 mV |

At 3C the SPMe reaches the cut-off after a tenth of the charge the DFN delivers:
its electrolyte correction breaks down where the electrolyte is depleted.
A DFN solve takes 4 to 7 s.

**Requirements.**

* The voltage within **20 mV** over the whole discharge at 1C and 2C, and within **50 mV** at 3C.
* The capacity at 3C within **2 %**.
* Say where in the cell, and when, your network's error is largest.

**What you hand in.** The voltage curves against the ground truth's at the three rates, the
electrolyte concentration at the end of the 3C discharge, the capacities, and
at what rate your network stops being worth training.

### MP10.1B · How much has the cell aged?

**The problem.** Section 6 of this set found one diffusivity from the surface concentration.
An aged cell loses lithium that can take part and diffuses more slowly, and
neither can be measured inside a sealed cell. What can be measured is its
voltage during a discharge. From the voltage alone, find the ageing - and so
how much capacity is left.

| | |
|---|---|
| cell | the LG M50, Chen2020, aged by two hidden factors: on the negative particles' diffusivity, and on the lithium they hold at full charge |
| case 1 | discharges at 0.5C, 1C and 2C, noise 5 mV |
| case 2 | 1C and 2C only, noise 5 mV |
| case 3 | 0.5C only, noise 10 mV |

**Why a plain PINN is not enough.** The voltage is one number at each instant for several unknowns, and the two
kinds of ageing bend the curve in similar ways. Which parameters a set of
discharges can tell apart is part of the answer.

**What you build.** An inverse PINN on the single-particle model with electrolyte: the two ageing
factors trainable, fitted to the physics and the measured voltages. You show
which parameter each discharge identifies.

**The ground truth you get.** `MP10.1B_readings.npz` to fit and `MP10.1B_truth.npz` with the hidden
factors, from the same script: PyBaMM's SPMe discharges of the aged cell.

**Worked example.** With the diffusivity halved and the lithium cut by 10 %, the capacity at 0.5C
falls from 5.015 to 4.455 Ah. On its own, the lost lithium moves the voltage by
up to 390 mV at 0.5C and 223 mV at 2C; the slower diffusion by up to 22 mV and
38 mV. The two effects' shapes correlate at 0.87 at 0.5C and 0.81 at 2C - the
higher rates are what tell them apart.

**Requirements.**

* Case 1: both ageing factors within **5 %**, and the capacity left within **1 %**.
* Case 3: say which factor one slow discharge cannot give you, and show it.

**What you hand in.** The recovered factors with uncertainties in each case, the capacity you predict
is left, and which discharge you would ask a test lab for next.